# Construct VOCAB table

*Please note that dfidf gets added to VOCAB in 04-bow-tfidf.ipynb*

## Setup

### Import Libraries

In [22]:
# import libraries
import pandas as pd
import numpy as np
import glob
import os
import re
import nltk
import xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup
from nltk.stem.porter import PorterStemmer

In [23]:
nltk_resources = [
    'punkt_tab',
    'averaged_perceptron_tagger_eng',
    'stopwords',
    'tagsets'
]

for resource in nltk_resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Could not download {resource}: {e}")

### Configuration and Data Loading

In [24]:
# declare OHCO and load in the corpus
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)

## Extract VOCAB

Features:
- n, p, i, n_chars

In [25]:
# extract VOCAB from CORPUS and get counts
VOCAB = CORPUS.term_str.value_counts().to_frame('n').sort_index()

# derive number of characters in terms
VOCAB['n_chars'] = VOCAB.index.str.len()

# derive probability (likelihood of a token appearing in the text)
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()

# derive information (how surprising or informative a term is)
VOCAB['i'] = -np.log2(VOCAB.p)

## Annotate VOCAB

Features:
- max_pos, max_pos_group
- n_pos, n_pos_group (pos ambiguity)
- stop (stopwords)
- porter stem
- dfidf (gets added later in pipeline)
- add ngrams if time permits?

In [26]:
# get max POS (most frequently associated part of speech category for each word)

# max pos per term (count every pair, unstack pos tags into columns, find column name with highest count per row)
VOCAB['max_pos'] = CORPUS[['term_str', 'pos']].value_counts().unstack(fill_value=0).idxmax(1)

# assign term a max pos group
VOCAB['max_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)

In [27]:
VOCAB['lemma'] = CORPUS.groupby('term_str')['lemma'].agg(lambda x: x.mode()[0])

In [28]:
# compute POS ambiguity (how many pos, by group, are associated with each term in the corpus)

# 
VOCAB['n_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack().count(1)
# VOCAB['cat_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().to_frame('n').reset_index()\
#     .groupby('term_str').pos_group.apply(lambda x: set(x))
VOCAB['n_pos'] = CORPUS[['term_str','pos']].value_counts().unstack().count(1)
# VOCAB['cat_pos'] = CORPUS[['term_str','pos']].value_counts().to_frame('n').reset_index()\
#     .groupby('term_str').pos.apply(lambda x: set(x))

# TODO FIX REDUNDANT COMPUTATION?

In [29]:
VOCAB.sample(10)

,n,n_chars,p,i,max_pos,max_pos_group,lemma,n_pos_group,n_pos
term_str,,,,,,,,,
recognize,62,9,0.000073,13.735375,VB,VB,recognize,1,2
comically,4,9,0.000005,17.689571,RB,RB,comically,1,1
hayward,1,7,0.000001,19.689571,NNP,NN,hayward,1,1
resplendentterrible,1,19,0.000001,19.689571,NNP,NN,resplendent⁠—terrible,1,1
pallor,8,6,0.000009,16.689571,NN,NN,pallor,1,1
retort,4,6,0.000005,17.689571,NN,NN,retort,1,1
evidently,103,9,0.000122,13.003071,RB,RB,evidently,1,1
particularly,71,12,0.000084,13.539824,RB,RB,particularly,1,1
develops,1,8,0.000001,19.689571,VBZ,VB,develop,1,1


In [30]:
VOCAB.loc[['s','t','don','ve','ll']]

,n,n_chars,p,i,max_pos,max_pos_group,lemma,n_pos_group,n_pos
term_str,,,,,,,,,
s,7970,1,0.009426,6.729207,VBZ,VB,’s,7,10
t,5,1,0.000006,17.367643,NNP,NN,t.,2,2
don,5,3,0.000006,17.367643,VB,VB,don,2,3
ve,1349,2,0.001595,9.291897,VBP,VB,’ve,2,4
ll,884,2,0.001045,9.901669,MD,MD,’ll,2,2


## Add Stopwords

Using NLTK's built in list for English and choosing to add 'said' to stopword list as it functions mostly as a dialogue marker rather than a content-bearing term. I am also adding contraction fragments.

In [31]:
# get stopwords from NLTK's built in list for English
sw = set(nltk.corpus.stopwords.words('english'))
# add dialogue marker to stopwords
# sw.update(['said']) # use .update() to add elements from an iterable into an existing set in place
sw.update(['said', 'couldn', 'didn', 'wouldn', 'shouldn', 'isn', 'aren', 'weren', 'haven', 'hadn', 'doesn', 'won', 't', 're', 've', 'll', 'd', 'm', 's']) # use .update() to add elements from an iterable into an existing set in place
VOCAB['stop'] = VOCAB.index.isin(sw)

In [32]:
# fragments = ['t', 're', 've', 'll', 'd', 'm', 's', 'couldn', 'didn', 'wouldn', 'shouldn', 'isn', 'aren', 'weren', 'haven', 'hadn', 'doesn', 'won']
# VOCAB[VOCAB.index.isin(fragments)]

In [33]:
# check high freq terms not flagged as stopwords
# VOCAB[VOCAB.stop == False].sort_values('n', ascending=False).head(50)

In [34]:
# check flagged stopwords for things that shouldn't be in stopwords

In [35]:
# VOCAB[VOCAB.stop == True].sort_values('n', ascending=False).head(30)

In [36]:
# VOCAB[VOCAB.stop == 1]

In [37]:
print("Stopwords:", int(VOCAB.stop.sum()))

Stopwords: 138


## Add Porter Stem

In [38]:
# instantiate stemmer (outside the lambda function)
porter = PorterStemmer()

VOCAB['porter_stem'] = VOCAB.index.map(lambda x: porter.stem(x))

In [39]:
# VOCAB

In [40]:
# check
VOCAB.columns.tolist()

['n',
 'n_chars',
 'p',
 'i',
 'max_pos',
 'max_pos_group',
 'lemma',
 'n_pos_group',
 'n_pos',
 'stop',
 'porter_stem']

In [41]:
# check lemma
VOCAB.loc[['frightened', 'murdered', 'suspicious', 'running'], ['lemma', 'max_pos']]

,lemma,max_pos
term_str,,
frightened,frightened,JJ
murdered,murder,VBN
suspicious,suspicious,JJ
running,run,VBG


Notice that adjectives don't change.

## Save Output

In [42]:
# save the VOCAB table to csv
VOCAB.to_csv('data/VOCAB.csv', sep='\t', index=True)